# What is Gymnasium and why are we using it?



[**Gymnasium**](https://gymnasium.farama.org/index.html) is a standard library for building reinforcement learning environments. It provides a clear structure for describing how an agent interacts with an environment over time.

Let's look at an example environment: CartPole. In this environment, a pole is attached to a cart that moves along a track. The agent's goal is to keep the pole balanced by applying forces to the cart.

<img src="https://gymnasium.farama.org/_images/cart_pole.gif" alt="cartpole-gif" width="200" />

CartPole is already implemented in Gymnasium, and we can initialize it using the `make` function.

In [ ]:
import gymnasium as gym

env = gym.make("CartPole-v1")
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>

We can see that this is a CartPole environment, wrapped with a few additional functionalities (time limit, auto reset, etc.). We will not be concerned with these details for now.

A Gymnasium environment defines:
* an **observation space** (of type `gymnasium.spaces.Space`) describing what the agent can observe
* an **action space** (of type `gymnasium.spaces.Space`) describing what the agent can do
* a `reset` method for starting a new episode
* a `step` method for applying an action and returning the result
 
 The `step(action)` function applies an action (which must belong to the `action_space`) and returns a tuple `(obs, reward, terminated, truncated, info)`, where `obs` is the next observation, `reward` is a scalar signal reflecting the immediate outcome of the action, `terminated` indicates whether the episode ended due to the task internal logic (e.g., goal reached or failure), and `truncated` indicates whether the episode was stopped due to an external constraint such as a time limit. The `info` dictionary can be used for additional diagnostics.

 Let's see how this works in practice by interacting with the CartPole environment:

In [ ]:
print(env.observation_space)
print('observation space: cart position, cart velocity, pole angle, pole angular velocity')
print(env.action_space)
print('action space: 0 (push cart to the left), 1 (push cart to the right)')

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
observation space: cart position, cart velocity, pole angle, pole angular velocity
Discrete(2)
action space: 0 (push cart to the left), 1 (push cart to the right)


In [ ]:
# SELECT AN ACTION:
# * 0 - push cart to the left
# * 1 - push cart to the right
# NOTE: existing cart velocity may still cause the cart to move in the opposite direction.
action = 0

obs0, info = env.reset()
print('initial observation:', obs0)

obs, reward, terminated, truncated, info = env.step(action)
print('next observation:', obs)

pos_diff = obs[0] - obs0[0]
print('cart position diff:', pos_diff, f"(moved {'left' if pos_diff < 0 else 'right'})")

initial observation: [-0.01680857  0.03133427  0.02573418  0.00337671]
next observation: [-0.01618188 -0.16414711  0.02580171  0.30406672]
cart position diff: 0.0006266851 (moved right)


Together, these components define the transition dynamics and reward function of the underlying decision process, meaning that the environment effectively implements the MDP (or POMDP) that the agent is learning to solve. This enables us to run simulations of the environment and to evaluate the performance of different agents consistently. For example, we can run a random agent in the CartPole environment and see how it performs:

In [ ]:
obs, info = env.reset()
total_reward = 0
steps_alive = 0
done = trunc = False
while not (done or trunc):
    action = env.action_space.sample()  # take a random action
    obs, reward, done, trunc, info = env.step(action)
    total_reward += reward
    steps_alive += 1
print(f"Total reward: {total_reward}")
print(f"Steps alive: {steps_alive}")
print("Episode ended due to:", "pole fell over" if done else "timeout")

Total reward: 11.0
Steps alive: 11
Episode ended due to: pole fell over
